# 04 — Consensus Disease Gene List

Aggregate DE results across disease-enriched subclusters (and datasets)
to produce consensus gene lists of genes consistently upregulated in
disease.

**Workflow:**
1. Load per-subcluster DE tables from one or more datasets
2. Filter to significantly upregulated genes
3. Build consensus gene list by **overlap** or **ranked scoring**
4. Run functional enrichment on the consensus list
5. Export consensus genes

**Scoring methods:**
- `"overlap"` — include genes upregulated in ≥ `MIN_OVERLAP` fraction of subclusters
- `"ranked"` — score genes by mean $-\log_{10}(p_{adj}) \times \log_2FC$ across
  subclusters, then threshold by `TOP_N_RANKED`

## 1. Import Libraries

In [ ]:
from __future__ import annotations

import math
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import scutils

warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Configuration

**Edit the cell below** before running the rest of the notebook.

Key parameters:
- `RESULTS_DIRS` — dataset output directories containing `de/` subfolders
- `SCORING_METHOD` — `"overlap"` or `"ranked"`
- `MIN_OVERLAP` — fraction of subclusters a gene must appear in (for `"overlap"` method)
- `TOP_N_RANKED` — number of top genes to retain (for `"ranked"` method)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input ─────────────────────────────────────────────────────────────────────
RESULTS_DIRS: list[Path] = [
    Path("results/disease_subclusters/dataset_A"),
    # Path("results/disease_subclusters/dataset_B"),
]

# ── DE significance thresholds ────────────────────────────────────────────────
PVAL_CUTOFF  = 0.05
LFC_CUTOFF   = 0.5
MIN_DE_GENES = 10    # minimum upregulated genes to include a subcluster

# ── Consensus method ──────────────────────────────────────────────────────────
# "overlap" — gene must be upregulated in ≥ MIN_OVERLAP fraction of subclusters
# "ranked"  — score = mean(-log10(padj) × log2FC), keep TOP_N_RANKED genes
SCORING_METHOD: str = "overlap"

# Overlap method parameters
MIN_OVERLAP: float = 0.5    # fraction of subclusters (0.0 = union, 1.0 = intersection)

# Ranked method parameters
TOP_N_RANKED: int = 200     # number of top-ranked genes to keep

# ── Enrichment settings ───────────────────────────────────────────────────────
ENRICHMENT_SOURCES: list[str] = [
    "GO:BP", "GO:CC", "GO:MF", "REAC", "WP", "KEGG",
]
ENRICHMENT_PVAL = 0.05
MAX_PATHWAYS    = 15
ORGANISM        = "hsapiens"

PATHWAY_COLORS = {
    "GO:BP": "#1f77b4",
    "GO:MF": "#ff7f0e",
    "GO:CC": "#2ca02c",
    "KEGG":  "#d62728",
    "REAC":  "#9467bd",
    "WP":    "#CD853F",
}

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("results/disease_subclusters/consensus")
SAVE_FIGS  = True

print("Configuration loaded.")
print(f"  Result dirs    : {[str(p) for p in RESULTS_DIRS]}")
print(f"  Scoring method : {SCORING_METHOD}")
print(f"  MIN_OVERLAP    : {MIN_OVERLAP}" if SCORING_METHOD == "overlap" else f"  TOP_N_RANKED: {TOP_N_RANKED}")
print(f"  Output dir     : {OUTPUT_DIR}")

## 3. Load DE Results

In [ ]:
all_de: list[pd.DataFrame] = []
upreg_sets: dict[str, set[str]] = {}

for results_dir in RESULTS_DIRS:
    de_dir = results_dir / "de"
    csv_files = sorted(de_dir.glob("de_*.csv"))
    # Exclude combined files
    csv_files = [f for f in csv_files if "all_subclusters" not in f.name]

    if not csv_files:
        print(f"⚠  No DE files found in {de_dir}")
        continue

    dataset_name = results_dir.name
    print(f"Dataset: {dataset_name} ({len(csv_files)} files)")

    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        label = csv_path.stem.removeprefix("de_")
        key = f"{dataset_name}/{label}" if len(RESULTS_DIRS) > 1 else label

        df["source_subcluster"] = key
        df["source_dataset"] = dataset_name
        all_de.append(df)

        mask = (df["padj"] < PVAL_CUTOFF) & (df["log2FoldChange"] > LFC_CUTOFF)
        upreg_sets[key] = set(df.loc[mask, "gene"].dropna().unique())
        print(f"  {key}: {len(upreg_sets[key])} upregulated genes")

combined_de = pd.concat(all_de, ignore_index=True) if all_de else pd.DataFrame()
n_subclusters = len(upreg_sets)

# Filter out subclusters with fewer than MIN_DE_GENES upregulated genes
skipped = {k: len(v) for k, v in upreg_sets.items() if len(v) < MIN_DE_GENES}
if skipped:
    print(f"\n⚠  Skipping {len(skipped)} subcluster(s) with < {MIN_DE_GENES} upregulated genes:")
    for k, n in skipped.items():
        print(f"    {k}: {n} genes")
    upreg_sets = {k: v for k, v in upreg_sets.items() if len(v) >= MIN_DE_GENES}
    n_subclusters = len(upreg_sets)

all_genes = set().union(*upreg_sets.values()) if upreg_sets else set()
print(f"\n✓ {n_subclusters} subclusters, {len(all_genes)} unique upregulated genes.")

### UpSet Plot — Visualise Gene Overlaps Across Subclusters

The UpSet plot shows which upregulated genes are shared between
subclusters and which are unique.

In [ ]:
if len(upreg_sets) >= 2:
    fig = scutils.pl.upset_plot(
        upreg_sets,
        figsize=(12, 6),
        sort_by="cardinality",
        show_counts=True,
        min_subset_size=1,
    )
    fig.suptitle(
        f"Overlap of upregulated genes  "
        f"(padj < {PVAL_CUTOFF}, log2FC > {LFC_CUTOFF})",
        fontsize=12,
        y=1.02,
    )

    if SAVE_FIGS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig_path = OUTPUT_DIR / "upset_upregulated_subclusters.png"
        fig.savefig(fig_path, bbox_inches="tight")
        print(f"Saved: {fig_path}")

    plt.show()
    plt.close(fig)
else:
    print("⚠  Need ≥ 2 subclusters for UpSet plot — skipping.")

## 4. Build Consensus Gene List

In [ ]:
if SCORING_METHOD == "overlap":
    # ── Overlap method ─────────────────────────────────────────────────
    gene_counts = Counter(
        gene for genes in upreg_sets.values() for gene in genes
    )
    min_count = max(math.ceil(MIN_OVERLAP * n_subclusters), 1)

    consensus_genes = sorted(
        gene for gene, cnt in gene_counts.items() if cnt >= min_count
    )

    print(f"Method         : overlap")
    print(f"MIN_OVERLAP    : {MIN_OVERLAP}")
    print(f"Num subclusters: {n_subclusters}")
    print(f"Min count      : {min_count}")
    print(f"Consensus genes: {len(consensus_genes)}")

elif SCORING_METHOD == "ranked":
    # ── Ranked scoring method ──────────────────────────────────────────
    # Filter to significant upregulated genes
    sig = combined_de[
        (combined_de["padj"] < PVAL_CUTOFF)
        & (combined_de["log2FoldChange"] > LFC_CUTOFF)
    ].copy()

    # Compute per-gene score: -log10(padj) × log2FC
    sig["score"] = -np.log10(sig["padj"].clip(lower=1e-300)) * sig["log2FoldChange"]

    # Mean score across subclusters (equal weight per subcluster)
    gene_scores = (
        sig.groupby("gene")["score"]
        .mean()
        .sort_values(ascending=False)
    )

    consensus_genes = gene_scores.head(TOP_N_RANKED).index.tolist()

    print(f"Method          : ranked")
    print(f"TOP_N_RANKED    : {TOP_N_RANKED}")
    print(f"Consensus genes : {len(consensus_genes)}")

    # Show top 20 with scores
    print("\nTop 20 genes by score:")
    display(gene_scores.head(20).to_frame("mean_score").style.format("{:.2f}"))

else:
    raise ValueError(f"Unknown SCORING_METHOD: {SCORING_METHOD!r}")

# Display
if len(consensus_genes) <= 100:
    print(f"\nGenes: {', '.join(consensus_genes)}")
else:
    print(f"\nFirst 100: {', '.join(consensus_genes[:100])}")
    print(f"  … and {len(consensus_genes) - 100} more")

## 5. Functional Enrichment of Consensus Genes

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not consensus_genes:
    print("⚠  Consensus gene set is empty — skipping enrichment.")
else:
    print(f"Querying g:Profiler for {len(consensus_genes)} consensus genes …")

    gdf = scutils.tl.get_enriched_terms(
        consensus_genes,
        sources=ENRICHMENT_SOURCES,
        pval_adjust_sign=ENRICHMENT_PVAL,
        organism=ORGANISM,
    )

    print(f"Found {len(gdf)} significant terms.")

    if not gdf.empty:
        fig = scutils.pl.create_pathway_dotplot(
            data=gdf,
            source_colors=PATHWAY_COLORS,
            max_pathways=MAX_PATHWAYS,
            figsize=(12, 10),
            title="Consensus gene list — enriched pathways",
        )

        if SAVE_FIGS:
            fig_path = OUTPUT_DIR / "consensus_gprofiler_dotplot.png"
            fig.savefig(fig_path, bbox_inches="tight")
            print(f"Saved: {fig_path}")

        plt.show()
        plt.close(fig)

        # Display table
        display_cols = [
            "source", "name", "p_value",
            "term_size", "intersection_size", "intersection_pct",
        ]
        display_cols = [c for c in display_cols if c in gdf.columns]

        display(
            gdf[display_cols].sort_values("p_value")
            .style
            .format({"p_value": "{:.2e}", "intersection_pct": "{:.1f}%"})
            .background_gradient(subset=["p_value"], cmap="Blues_r")
        )

        # Save enrichment
        save_df = gdf.copy()
        if "parents" in save_df.columns:
            save_df["parents"] = save_df["parents"].apply(
                lambda p: "|".join(p) if isinstance(p, list) else p
            )
        save_df.to_csv(OUTPUT_DIR / "consensus_enrichment.csv", index=False)
        print(f"Saved: {OUTPUT_DIR / 'consensus_enrichment.csv'}")

## 6. Export Consensus Gene List

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Plain text gene list
genes_path = OUTPUT_DIR / "consensus_genes.txt"
genes_path.write_text("\n".join(consensus_genes) + "\n")
print(f"Saved {len(consensus_genes)} consensus genes → {genes_path}")

# CSV with per-gene statistics
if SCORING_METHOD == "ranked" and not combined_de.empty:
    gene_stats = (
        combined_de[
            (combined_de["padj"] < PVAL_CUTOFF)
            & (combined_de["log2FoldChange"] > LFC_CUTOFF)
            & (combined_de["gene"].isin(consensus_genes))
        ]
        .groupby("gene")
        .agg(
            n_subclusters=("source_subcluster", "nunique"),
            mean_log2FC=("log2FoldChange", "mean"),
            median_padj=("padj", "median"),
        )
        .sort_values("mean_log2FC", ascending=False)
    )
    stats_path = OUTPUT_DIR / "consensus_genes_stats.csv"
    gene_stats.to_csv(stats_path)
    print(f"Saved gene statistics → {stats_path}")

elif SCORING_METHOD == "overlap":
    gene_counts_df = pd.DataFrame(
        [(g, gene_counts[g]) for g in consensus_genes],
        columns=["gene", "n_subclusters"],
    ).sort_values("n_subclusters", ascending=False)
    stats_path = OUTPUT_DIR / "consensus_genes_stats.csv"
    gene_counts_df.to_csv(stats_path, index=False)
    print(f"Saved gene statistics → {stats_path}")

print(f"\n✓ All outputs saved to {OUTPUT_DIR}")

---

## Summary

| Step | Function | Key parameters |
|------|----------|----------------|
| Load DE results | `pd.read_csv()` | `RESULTS_DIRS` |
| Consensus (overlap) | `collections.Counter` + threshold | `MIN_OVERLAP` |
| Consensus (ranked) | mean($-\log_{10}p \times \text{LFC}$) | `TOP_N_RANKED` |
| g:Profiler enrichment | `scutils.tl.get_enriched_terms()` | `ENRICHMENT_SOURCES`, `ORGANISM` |
| Pathway dot-plot | `scutils.pl.create_pathway_dotplot()` | `MAX_PATHWAYS` |
| Export | `Path.write_text()`, `pd.to_csv()` | `OUTPUT_DIR` |

**Next:** Run `05_gene_scoring.ipynb` to score cells using the consensus
gene list via ULM.